# Walmart Store Sales Forecasting — ARIMA / SARIMAX

In [ ]:
!pip install kaggle pmdarima statsmodels -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
! chmod 600 /root/.kaggle/kaggle.json
! kaggle competitions download Walmart-Recruiting-Store-Sales-Forecasting
! unzip -o Walmart-Recruiting-Store-Sales-Forecasting.zip
! unzip -o train.csv.zip
! unzip -o test.csv.zip
! unzip -o features.csv.zip

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import pmdarima as pm
from joblib import Parallel, delayed

from sklearn.base import BaseEstimator, TransformerMixin

pd.set_option("display.max_columns", 60)
plt.rcParams["figure.dpi"] = 110

## 1. Load and merge data

In [ ]:
TRAIN_PATH    = "train.csv"
FEATURES_PATH = "features.csv"
STORES_PATH   = "stores.csv"

train_raw    = pd.read_csv(TRAIN_PATH,    parse_dates=["Date"])
features_raw = pd.read_csv(FEATURES_PATH, parse_dates=["Date"])
stores_raw   = pd.read_csv(STORES_PATH)

df_merged = (
    train_raw
    .merge(features_raw, on=["Store", "Date", "IsHoliday"], how="left")
    .merge(stores_raw,   on=["Store"],                       how="left")
)
df_merged = df_merged.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)

print(f"Shape      : {df_merged.shape}")
print(f"Date range : {df_merged['Date'].min().date()} -> {df_merged['Date'].max().date()}")
print(f"Store-Dept combinations: {df_merged.groupby(['Store','Dept']).ngroups}")

## 2. Exogenous features

In [ ]:
SUPERBOWL_DATES = pd.to_datetime(["2010-02-12", "2011-02-11", "2012-02-10", "2013-02-08"])
LABORDAY_DATES  = pd.to_datetime(["2010-09-10", "2011-09-09", "2012-09-07", "2013-09-06"])
THANKSGIVING_DATES = pd.to_datetime(["2010-11-26", "2011-11-25", "2012-11-23", "2013-11-29"])
CHRISTMAS_DATES = pd.to_datetime(["2010-12-31", "2011-12-30", "2012-12-28", "2013-12-27"])


class CalendarFeatureTransformer(BaseEstimator, TransformerMixin):
    """Same as in the tree-model notebook -- reused unchanged as SARIMAX exog."""

    def fit(self, X, y=None):
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        X = X.copy()
        X["Date"] = pd.to_datetime(X["Date"])
        week = X["Date"].dt.isocalendar().week.astype(int)
        X["week_sin"] = np.sin(2 * np.pi * week / 52)
        X["week_cos"] = np.cos(2 * np.pi * week / 52)
        X["is_thanksgiving"] = X["Date"].isin(THANKSGIVING_DATES).astype(int)
        X["is_christmas"]    = X["Date"].isin(CHRISTMAS_DATES).astype(int)
        X["is_superbowl"]    = X["Date"].isin(SUPERBOWL_DATES).astype(int)
        X["is_laborday"]     = X["Date"].isin(LABORDAY_DATES).astype(int)
        return X


class MarkdownFeatureTransformer(BaseEstimator, TransformerMixin):
    """Same as in the tree-model notebook, collapsed to one exog signal."""

    def fit(self, X, y=None):
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        X = X.copy()
        mdc = [c for c in ["MarkDown1","MarkDown2","MarkDown3","MarkDown4","MarkDown5"]
               if c in X.columns]
        if mdc:
            X["total_markdown_log"] = np.log1p(X[mdc].fillna(0).sum(axis=1))
        return X


def fourier_terms(dates: pd.Series, period: int = 52, order: int = 3) -> pd.DataFrame:
    """
    Fourier-series exogenous regressors -- a standard substitute for a true
    seasonal(m=52) SARIMA term, which is too slow for auto_arima to fit.
    Encodes the annual cycle with `order` sine/cosine pairs instead of 52 lags.
    """
    t = (dates - dates.min()).dt.days.values / 7.0  # week index
    out = {}
    for k in range(1, order + 1):
        out[f"fourier_sin_{k}"] = np.sin(2 * np.pi * k * t / period)
        out[f"fourier_cos_{k}"] = np.cos(2 * np.pi * k * t / period)
    return pd.DataFrame(out, index=dates.index)


EXOG_COLS = [
    "week_sin", "week_cos", "is_thanksgiving", "is_christmas",
    "is_superbowl", "is_laborday", "total_markdown_log", "IsHoliday",
]

## 3. Train/val split

In [ ]:
VAL_START = "2012-04-01"
H = 4  # forecast horizon -- matches NeuralForecast setup (H=4 weeks)

calendar_tf = CalendarFeatureTransformer()
markdown_tf = MarkdownFeatureTransformer()

df_fe = calendar_tf.transform(df_merged)
df_fe = markdown_tf.transform(df_fe)
fcols = fourier_terms(df_fe["Date"], period=52, order=3)
df_fe = pd.concat([df_fe, fcols], axis=1)
EXOG_COLS_FULL = EXOG_COLS + list(fcols.columns)

print(f"Exogenous regressors ({len(EXOG_COLS_FULL)}): {EXOG_COLS_FULL}")

## 4. Stationarity check and (p,d,q) selection demo

In [ ]:
def get_series(df: pd.DataFrame, store: int, dept: int) -> pd.DataFrame:
    s = (
        df[(df["Store"] == store) & (df["Dept"] == dept)]
        .sort_values("Date")
        .set_index("Date")
    )
    s = s.asfreq("W-FRI")  # enforce regular weekly spacing; introduces NaN on gaps
    s["Weekly_Sales"] = s["Weekly_Sales"].interpolate(limit_direction="both")
    for c in EXOG_COLS_FULL:
        s[c] = s[c].ffill().bfill()
    return s


DEMO_STORE, DEMO_DEPT = 1, 1
demo = get_series(df_fe, DEMO_STORE, DEMO_DEPT)

adf_stat, adf_p, *_ = adfuller(demo["Weekly_Sales"].dropna())
print(f"ADF statistic: {adf_stat:.3f}   p-value: {adf_p:.4f}")
print("-> p < 0.05 means stationary at this level (no further differencing needed);"
      " otherwise auto_arima will pick d>=1.")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
plot_acf(demo["Weekly_Sales"].dropna(), lags=60, ax=axes[0])
plot_pacf(demo["Weekly_Sales"].dropna(), lags=60, ax=axes[1])
plt.tight_layout()
plt.show()

decomp = seasonal_decompose(demo["Weekly_Sales"].dropna(), period=52, model="additive")
decomp.plot()
plt.tight_layout()
plt.show()

## 5. Fit SARIMAX per series

In [ ]:
def fit_and_forecast_one_series(df_fe: pd.DataFrame, store: int, dept: int,
                                 val_start: str, h: int) -> dict:
    """
    Fits auto_arima with Fourier exog on one (Store, Dept) series and forecasts
    the validation window. Returns metrics + predictions for later aggregation.
    """
    s = get_series(df_fe, store, dept)
    s_train = s[s.index < val_start]
    s_val   = s[s.index >= val_start].iloc[:h]  # first h weeks of validation

    if len(s_train) < 60 or s_val.empty:
        return None  # not enough history for this Store-Dept pair

    y_train    = s_train["Weekly_Sales"]
    exog_train = s_train[EXOG_COLS_FULL]
    exog_val   = s_val[EXOG_COLS_FULL]

    try:
        model = pm.auto_arima(
            y_train,
            exogenous=exog_train,
            seasonal=False,        # seasonality handled via Fourier exog, not native m=52
            stepwise=True,
            suppress_warnings=True,
            error_action="ignore",
            max_p=5, max_q=5, max_d=2,
        )
        preds = model.predict(n_periods=len(s_val), exogenous=exog_val)
    except Exception as e:
        return {"store": store, "dept": dept, "error": str(e)}

    weights = np.where(s_val["IsHoliday"].values, 5.0, 1.0)
    wmae_val = np.sum(weights * np.abs(s_val["Weekly_Sales"].values - preds)) / np.sum(weights)

    return {
        "store": store, "dept": dept,
        "order": model.order,
        "wmae": wmae_val,
        "n_train": len(s_train),
        "y_true": s_val["Weekly_Sales"].values,
        "y_pred": preds,
        "dates": s_val.index,
    }

## 6. Loop over multiple (Store, Dept) pairs

In [ ]:
N_SERIES = 20  # None = ყველა store-dept წყვილი (ნელი! გამოიყენეთ n_jobs პარალელიზაციისთვის)

vol = (
    df_fe.groupby(["Store", "Dept"])["Weekly_Sales"].mean()
    .sort_values(ascending=False)
)
pairs = list(vol.index[:N_SERIES]) if N_SERIES else list(vol.index)
print(f"სერიების რაოდენობა: {len(pairs)}")

results = Parallel(n_jobs=-1, verbose=5)(
    delayed(fit_and_forecast_one_series)(df_fe, store, dept, VAL_START, H)
    for store, dept in pairs
)
results = [r for r in results if r is not None and "error" not in r]
print(f"წარმატებით დაფიტდა: {len(results)} / {len(pairs)} სერია")

## 7. Aggregated WMAE

In [ ]:
all_true = np.concatenate([r["y_true"] for r in results])
all_pred = np.concatenate([r["y_pred"] for r in results])
all_hol  = np.concatenate([
    (df_fe.set_index(["Store","Dept"]).loc[(r["store"], r["dept"])]
       .set_index("Date").loc[r["dates"], "IsHoliday"].values)
    for r in results
])
weights = np.where(all_hol, 5.0, 1.0)
overall_wmae = np.sum(weights * np.abs(all_true - all_pred)) / np.sum(weights)

per_series = pd.DataFrame([
    {"Store": r["store"], "Dept": r["dept"], "order": r["order"], "wmae": r["wmae"]}
    for r in results
]).sort_values("wmae")

print(f"Overall WMAE (SARIMAX + Fourier exog, {len(results)} series): {overall_wmae:,.2f}")
display(per_series.head(10))
display(per_series.tail(10))

## 8. Guide for running on the full dataset